# Disease-Gene Similarity-Based Drug Recommendation (CBF)
**SCE331 Data Mining · Team Project · Revised implementation (2026-11-14)**

**Research question:** Can similarities between disease gene profiles recover known drug–disease indication links hidden from training?

**Method:** disease–gene weighted profile → cosine similarity → top-k neighbor diseases → training indication links → disease-specific drug ranking.

이 파일은 기존 CBF A 방식을 유지하면서 다음 네 가지 피드백을 반영한 **학생 팀 프로젝트 구현본**이다.

1. **Pure/Strict CBF를 주 평가**로 사용한다. Popularity fallback과 유전자 정보 보유 질병만의 결과는 따로 보고한다.
2. `raw`/`idf`, 이웃 질병 수 `k`, 이웃 유사도 가중 지수 `p=1,2`를 **Validation에서만** 선택한다.
3. `preliminary`와 `team_final` 실행 모드를 명시하고, 최종 모드의 필수 공통 파일이 없으면 **실행을 중단**한다. 후보 약물 카탈로그를 정답 분할과 구분한다.
4. **5개 분할**로 파라미터 및 성능 안정성을 점검한다. 단, `team_final`에서는 공통 Test를 고정하고 **학습 데이터 내부에서만 5회 Validation**한 다음 공통 Test를 한 번 평가한다.

**범위:** OptimusKG V3.2; 유전자 연관성 score ≥ 0.3. 외부 데이터의 가용성 차이와 실제 약효는 검증하지 않는다. 원본·파생 관계 CSV의 재배포 권한은 별도 확인이 필요하다.

**강의 연계:** 09강의 Feature Vector, Cosine Similarity, TF-IDF/IDF. `raw`와 `idf`는 별개의 주요 알고리즘이 아니라 같은 CBF의 설정이다.

## 1. Configure the experiment
처음 실행할 때 `RUN_MODE = "preliminary"`로 둔다. 팀의 공통 데이터 범위·분할·후보 목록이 완성되면 `team_final`로 바꾼다.

- **preliminary:** 필수 CSV 두 개만 사용. 금기 충돌 339쌍을 제외하지 못하므로 **최종 성능 아님**. 각 분할의 약물 후보는 **Train에 나타난 약물만** 사용한다(정답에서 후보 목록 생성 방지).
- **team_final:** `team_eligible_indications.csv`(57,835 정제 적응증), `team_split.csv`, `team_candidate_drugs.csv`가 **모두 필수**. 공통 Test는 고정한다.
- 보조 허가 적응증 평가는 최종 모드에서 `RUN_SECONDARY = True`로 직접 지정하고 검증된 `team_approved_indications.csv`를 추가해야 한다.

모든 모델이 같은 후보 약물·평가 질병·정답 분할을 사용하도록 팀원들과 확인한다.

In [ ]:
from pathlib import Path
from collections import defaultdict
from math import log2
import hashlib
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

RUN_MODE = "preliminary"  # "preliminary" or "team_final"
RUN_SECONDARY = False
SEEDS = [42, 43, 44, 45, 46]
MIN_GENE_SCORE = 0.3
FEATURE_MODES = ["raw", "idf"]
NEIGHBOR_K_OPTIONS = [10, 20, 50]
SIM_POWER_OPTIONS = [1, 2]
METRIC_K = [5, 10, 20, 50, 100]
SELECTION_METRIC = "NDCG@10"
TOP_N = 100
EXPECTED_ELIGIBLE_PAIRS = 57835
EXPECTED_CONFLICT_EXCLUSIONS = 339

DATA_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
OUTPUT_DIR = DATA_DIR / "cbf_results_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert RUN_MODE in {"preliminary", "team_final"}
assert len(set(SEEDS)) == len(SEEDS) and len(SEEDS) == 5
assert TOP_N >= max(METRIC_K)
print("Run mode:", RUN_MODE)
print("Data directory:", DATA_DIR)
print("Five seeds:", SEEDS)
print("Settings:", len(FEATURE_MODES) * len(NEIGHBOR_K_OPTIONS) * len(SIM_POWER_OPTIONS))

## 2. Read the datasets and validate data scope
원본 CSV에 없는 충돌 339쌍은 임의로 추정해서 제외하지 않는다. 팀이 검증한 정제 목록만 사용한다.

**최종 모드 파일 형식**
- `team_eligible_indications.csv`: `disease_id,drug_id` (중복 없는 57,835쌍)
- `team_split.csv`: `disease_id,drug_id,split` (`train`, `validation`, `test`)
- `team_candidate_drugs.csv`: `drug_id` (학습/평가에 공통으로 사용할, 팀이 미리 정의한 약물 카탈로그)
- `team_approved_indications.csv`: `disease_id,drug_id` (선택 보조 평가용)

`disease_names.csv`, `drug_names.csv`는 출력용으로만 선택적으로 사용한다.

In [ ]:
required_files = ["disease_gene_t03.csv", "indication.csv"]
if RUN_MODE == "team_final":
    required_files += ["team_eligible_indications.csv", "team_split.csv", "team_candidate_drugs.csv"]
    if RUN_SECONDARY:
        required_files.append("team_approved_indications.csv")
elif RUN_SECONDARY:
    raise ValueError("Secondary evaluation requires RUN_MODE='team_final'.")
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Required files are missing: {missing}")

def normalize_pairs(frame):
    names = ["disease_id", "drug_id"]
    if not set(names).issubset(frame.columns):
        raise ValueError("Pair data must include disease_id and drug_id.")
    if frame[names].isna().any().any():
        raise ValueError("Pair data includes missing IDs.")
    pairs = frame[names].astype(str).copy()
    if pairs.duplicated().any():
        raise ValueError("Duplicate disease-drug pairs found.")
    return pairs.reset_index(drop=True)

def pair_set(frame):
    return set(frame[["disease_id", "drug_id"]].itertuples(index=False, name=None))

gene_raw = pd.read_csv(DATA_DIR / "disease_gene_t03.csv")
ind_raw = pd.read_csv(DATA_DIR / "indication.csv")
if not {"disease_id", "gene_id", "score"}.issubset(gene_raw.columns):
    raise ValueError("Disease-gene CSV must contain disease_id, gene_id, score.")
original_pairs = normalize_pairs(ind_raw)
gene_raw["score"] = pd.to_numeric(gene_raw["score"], errors="coerce")
gene_edges = gene_raw.dropna(subset=["disease_id", "gene_id", "score"])
gene_edges = gene_edges.loc[gene_edges["score"] >= MIN_GENE_SCORE].copy()
gene_edges = gene_edges.sort_values("score", ascending=False).drop_duplicates(["disease_id", "gene_id"])

if RUN_MODE == "team_final":
    positive_pairs = normalize_pairs(pd.read_csv(DATA_DIR / "team_eligible_indications.csv"))
    if not pair_set(positive_pairs).issubset(pair_set(original_pairs)):
        raise ValueError("Eligible indications contain unknown original indication pairs.")
    if len(positive_pairs) != EXPECTED_ELIGIBLE_PAIRS or len(original_pairs)-len(positive_pairs) != EXPECTED_CONFLICT_EXCLUSIONS:
        raise ValueError("Eligible indication count or 339-pair exclusion does not match team scope.")
    catalog = pd.read_csv(DATA_DIR / "team_candidate_drugs.csv", dtype=str)
    if "drug_id" not in catalog.columns or catalog["drug_id"].isna().any():
        raise ValueError("Team candidate catalog must contain nonmissing drug_id.")
    if catalog["drug_id"].duplicated().any():
        raise ValueError("Team candidate catalog contains duplicate drugs.")
    team_candidate_drugs = set(catalog["drug_id"])
    if not set(positive_pairs["drug_id"]).issubset(team_candidate_drugs):
        raise ValueError("The team catalog excludes eligible positive drugs.")
    label_status = "TEAM_ELIGIBLE_57835"
    catalog_status = "TEAM_FIXED_CATALOG"
else:
    positive_pairs = original_pairs.copy()
    team_candidate_drugs = None
    label_status = "PRELIMINARY_58174_UNFILTERED"
    catalog_status = "TRAIN_ONLY_PER_SPLIT"
    print("WARNING: the 339 contraindication-conflicting pairs are NOT excluded.")

print("Disease-gene links:", len(gene_edges))
print("Original indication pairs:", len(original_pairs))
print("Eligible indication pairs:", len(positive_pairs))
print("Excluded indication pairs:", len(original_pairs) - len(positive_pairs))
print("Indication diseases:", positive_pairs["disease_id"].nunique())
print("Label status:", label_status, "| Candidate policy:", catalog_status)
display(gene_raw.head(3))
display(ind_raw.head(3))

## 3. Create five independent splits without leaking Test labels
**preliminary:** 시드 42~46마다 별도의 Train/Validation/Test를 생성한다. `train`에 최소 2개의 관계를 남기되, 관계가 적은 질병은 Validation 또는 Test가 없을 수도 있다. 성능은 **5회 평균·표준편차**로 보고한다.

**team_final:** `team_split.csv`의 **공통 Test는 고정**하고, 팀 공식 Validation 한 번 + Train/Validation 영역 내부의 다른 네 가지 분할로 5회 Validation을 수행한다. 선택된 설정으로 **공통 Test를 단 한 번** 평가한다. 따라서 최종 모드의 5회 분석은 *5개의 독립적인 Test 결과*가 아니라 *5개의 내부 Validation 결과*임을 구분한다.

In [ ]:
def stable_hash(seed, disease_id, drug_id):
    key = f"{seed}|{disease_id}|{drug_id}".encode("utf-8")
    return int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), "big")

def generate_split(pairs, seed, include_test=True):
    rows = []
    for disease, group in pairs.groupby("disease_id", sort=True):
        ordered = sorted(group["drug_id"], key=lambda drug: (stable_hash(seed, disease, drug), drug))
        n = len(ordered)
        test_n = min(max(1, round(0.2*n)), n-2) if include_test and n >= 3 else 0
        val_n = min(max(1, round(0.2*n)), n-test_n-2) if n-test_n >= 3 else 0
        for i, drug in enumerate(ordered):
            split = "test" if i < test_n else ("validation" if i < test_n+val_n else "train")
            rows.append((disease, drug, split))
    return pd.DataFrame(rows, columns=["disease_id", "drug_id", "split"])

def validate_split(split, expected):
    required = {"disease_id", "drug_id", "split"}
    if not required.issubset(split.columns):
        raise ValueError("Split CSV needs disease_id, drug_id, split.")
    result = split[list(required)].copy()
    result[["disease_id", "drug_id", "split"]] = result[["disease_id", "drug_id", "split"]].astype(str)
    result["split"] = result["split"].str.strip().str.lower()
    if not set(result["split"]).issubset({"train", "validation", "test"}):
        raise ValueError("Split values must be train, validation, or test.")
    if result.duplicated(["disease_id", "drug_id"]).any():
        raise ValueError("Duplicate pairs in split data.")
    if pair_set(result) != pair_set(expected):
        raise ValueError("Split pairs do not exactly match eligible indication pairs.")
    return result[["disease_id", "drug_id", "split"]]

def split_parts(split):
    cols = ["disease_id", "drug_id"]
    return tuple(split.loc[split["split"]==label, cols].copy() for label in ("train", "validation", "test"))

runs = []
if RUN_MODE == "preliminary":
    for seed in SEEDS:
        split = validate_split(generate_split(positive_pairs, seed), positive_pairs)
        train, valid, test = split_parts(split)
        runs.append({"seed":seed, "train":train, "validation":valid, "test":test, "split":split})
else:
    official = validate_split(pd.read_csv(DATA_DIR / "team_split.csv"), positive_pairs)
    official_train, official_val, official_test = split_parts(official)
    if official_train.empty or official_val.empty or official_test.empty:
        raise ValueError("Team final split must have nonempty train, validation, and test.")
    full_dev = pd.concat([official_train, official_val], ignore_index=True)
    for i, seed in enumerate(SEEDS):
        if i == 0:
            train, valid = official_train.copy(), official_val.copy()
        else:
            internal = generate_split(full_dev, seed, include_test=False)
            train, valid, _ = split_parts(internal)
        runs.append({"seed":seed, "train":train, "validation":valid, "test":official_test, "split":official if i==0 else None})

for run in runs:
    train, valid, test = run["train"], run["validation"], run["test"]
    assert not (pair_set(train) & pair_set(valid))
    assert not (pair_set(train) & pair_set(test))
    assert not (pair_set(valid) & pair_set(test))
    assert pair_set(pd.concat([train, valid, test])) == pair_set(positive_pairs)

split_counts = pd.DataFrame([{"seed":r["seed"], "train":len(r["train"]), "validation":len(r["validation"]), "test":len(r["test"])} for r in runs])
display(split_counts)
print("Split protocol:", "5 independent preliminary tests" if RUN_MODE=="preliminary" else "1 fixed official test + 5 internal validation splits")

## 4. Build disease gene vectors and cosine similarity
각 질병의 `gene_id`별 `score`를 특징 벡터로 저장하고, **L2 정규화** 후 행렬 곱으로 코사인 유사도를 계산한다.

- `raw`: 기존 evidence score 그대로
- `idf`: 흔한 유전자 비중을 줄인 score × IDF
- 0 벡터(유전자 정보 없음)는 유사도 0으로 남는다. 이 질병을 평가에서 자동 제외하지 않는다.

`idf`는 별도의 주요 알고리즘이 아닌 CBF의 특징 가중 방식이다.

In [ ]:
disease_ids = pd.Index(sorted(positive_pairs["disease_id"].unique()))
gene_ids = pd.Index(sorted(gene_edges["gene_id"].unique()))
disease_to_row = {x:i for i,x in enumerate(disease_ids)}
subset = gene_edges.loc[gene_edges["disease_id"].isin(disease_ids)]
rows = disease_ids.get_indexer(subset["disease_id"])
cols = gene_ids.get_indexer(subset["gene_id"])
values = subset["score"].to_numpy(dtype=np.float64)
base = csr_matrix((values, (rows, cols)), shape=(len(disease_ids), len(gene_ids)))
feature_available = np.diff(base.indptr) > 0

df_gene = gene_edges.groupby("gene_id")["disease_id"].nunique().reindex(gene_ids, fill_value=0).to_numpy()
all_gene_diseases = gene_edges["disease_id"].nunique()
idf = np.log((all_gene_diseases+1)/(df_gene+1))+1
profile = {
    "raw": normalize(base, norm="l2", axis=1).tocsr(),
    "idf": normalize(base.multiply(idf).tocsr(), norm="l2", axis=1).tocsr(),
}
start_sim = time.perf_counter()
similarity = {mode:(matrix @ matrix.T).toarray().astype(np.float32) for mode,matrix in profile.items()}
for values in similarity.values():
    np.fill_diagonal(values, 0.0)
print("Feature matrix shape:", base.shape)
print("Diseases with gene features:", int(feature_available.sum()))
print("Diseases without gene features:", int((~feature_available).sum()))
print("Similarity matrix preparation (sec):", round(time.perf_counter()-start_sim, 2))

name_maps = {}
for name, idcol, outname in [("disease_names.csv", "disease_id", "disease"), ("drug_names.csv", "drug_id", "drug")]:
    path = DATA_DIR / name
    if path.is_file():
        names = pd.read_csv(path, dtype=str)
        name_maps[outname] = names.dropna(subset=[idcol]).drop_duplicates(idcol).set_index(idcol)["name"].to_dict()
    else:
        name_maps[outname] = {}

## 5. Implement Strict CBF and separate comparison methods
**Strict CBF:** 비슷한 질병의 학습용 적응증만으로 점수를 계산한다. `p=1`은 기존 유사도 가중합, `p=2`는 유사도가 높은 질병에 더 큰 비중을 주는 가중합이다.

- 주 평가: **Strict CBF (전체 Test 질병)**
- 보조 평가: **Strict CBF (유전자 정보 있는 질병만)**, **CBF + Popularity fallback (전체)**
- 대조 방법: **Popularity Baseline (전체)**

**중요:** CBF가 추천할 약물이 100개 미만일 때 Strict는 억지로 100개를 채우지 않는다. Fallback은 별도의 보조 방법으로만 보고한다. 후보 약물 카탈로그는 팀 최종 모드에서 미리 정의한 파일을 이용하고, 예비 모드에서는 매 분할의 Train 약물만 사용한다.

In [ ]:
def lookup_by_disease(pairs):
    return pairs.groupby("disease_id")["drug_id"].apply(set).to_dict()

def make_fold(train, candidate_catalog=None):
    lookup = lookup_by_disease(train)
    drug_counts = train["drug_id"].value_counts().to_dict()
    catalog = set(train["drug_id"]) if candidate_catalog is None else set(candidate_catalog)
    if not set(train["drug_id"]).issubset(catalog):
        raise ValueError("Training indication contains drugs outside candidate catalog.")
    popular = sorted((d for d in catalog if drug_counts.get(d, 0)>0), key=lambda d:(-drug_counts[d], d))
    sources = np.array([disease_to_row[d] for d in lookup if d in disease_to_row and feature_available[disease_to_row[d]]], dtype=int)
    sources.sort()
    return {"train_lookup":lookup, "popular":popular, "catalog":catalog, "source_rows":sources}

def find_neighbors(disease, mode, fold):
    target = disease_to_row.get(disease)
    if target is None or not feature_available[target]:
        return []
    possible = fold["source_rows"]
    sims = similarity[mode][target, possible]
    valid = (sims>0) & (possible != target)
    indices = possible[valid]
    strengths = sims[valid]
    sorted_idx = np.argsort(-strengths, kind="stable")[:max(NEIGHBOR_K_OPTIONS)]
    return [(str(disease_ids[indices[i]]), float(strengths[i])) for i in sorted_idx]

def neighbors_for_targets(targets, mode, fold):
    return {d:find_neighbors(d, mode, fold) for d in targets}

def rank_drugs(disease, neighbors, fold, k, power, fallback=False):
    known = fold["train_lookup"].get(disease, set())
    scores = defaultdict(float)
    for neighbor, sim in neighbors[:k]:
        w = sim ** power
        for drug in fold["train_lookup"].get(neighbor, set()):
            if drug in fold["catalog"] and drug not in known:
                scores[drug] += w
    sorted_drugs = sorted(scores, key=lambda d:(-scores[d], d))
    results = [(d, scores[d], "CBF") for d in sorted_drugs[:TOP_N]]
    if fallback and len(results)<TOP_N:
        used = set(d for d,_,_ in results) | known
        for drug in fold["popular"]:
            if drug not in used:
                results.append((drug, 0.0, "Popularity fallback"))
                used.add(drug)
                if len(results)==TOP_N:
                    break
    return results

def popular_rank(disease, fold):
    known = fold["train_lookup"].get(disease, set())
    return [(drug, 0.0, "Popularity") for drug in fold["popular"] if drug not in known][:TOP_N]

def recall_at_k(ranked, truth, k):
    return len(set(ranked[:k]) & truth) / len(truth) if truth else np.nan

def ndcg_at_k(ranked, truth, k):
    if not truth:
        return np.nan
    dcg = sum(1/log2(i+2) for i, drug in enumerate(ranked[:k]) if drug in truth)
    idcg = sum(1/log2(i+2) for i in range(min(len(truth), k)))
    return dcg/idcg if idcg else np.nan

def evaluate(truth_pairs, fold, method="strict", mode="raw", k=20, power=1, precomputed=None, return_details=False):
    truths = lookup_by_disease(truth_pairs)
    targets = sorted(truths)
    if precomputed is None and method!="popularity":
        precomputed = neighbors_for_targets(targets, mode, fold)
    output = []
    for disease in targets:
        if method=="popularity":
            info = popular_rank(disease, fold)
        else:
            info = rank_drugs(disease, precomputed[disease], fold, k, power, fallback=(method=="fallback"))
        ranked = [d for d,_,_ in info]
        truth = truths[disease]
        row = {"disease_id":disease, "n_truth":len(truth), "n_recommended":len(ranked),
               "has_gene_profile":bool(feature_available[disease_to_row[disease]]),
               "n_cbf_recommendations":sum(source=="CBF" for _,_,source in info),
               "n_fallback_recommendations":sum(source=="Popularity fallback" for _,_,source in info)}
        for cut in METRIC_K:
            row[f"Recall@{cut}"] = recall_at_k(ranked, truth, cut)
            row[f"NDCG@{cut}"] = ndcg_at_k(ranked, truth, cut)
        output.append(row)
    table = pd.DataFrame(output)
    return table if return_details else summarize_metrics(table)

def summarize_metrics(table):
    summary = {"eval_diseases":len(table),
               "gene_profile_fraction":table["has_gene_profile"].mean(),
               "recommendation_fraction":(table["n_recommended"]>0).mean(),
               "mean_cbf_slots":table["n_cbf_recommendations"].mean(),
               "mean_fallback_slots":table["n_fallback_recommendations"].mean()}
    for cut in METRIC_K:
        summary[f"Recall@{cut}"] = table[f"Recall@{cut}"].mean()
        summary[f"NDCG@{cut}"] = table[f"NDCG@{cut}"].mean()
    return summary

print("Strict CBF, fallback, and popularity baseline functions are ready.")

## 6. Compare hyperparameters using five Validation splits (no Test)
설정 12개 = `raw`/`idf` × `k=10,20,50` × `sim_power=1,2`.

**Strict CBF의 Validation NDCG@10 평균**으로 1개 설정을 선택한다. 표준편차로 분할별 민감도를 함께 보고한다. Test로 설정을 고르지 않는다. 동점일 때는 Recall@10 평균이 높은 설정, 그다음 단순한 설정을 선택한다.

최종 모드에서는 팀 공식 Validation을 첫 번째로 사용하고 추가 4회는 공통 Test를 제외한 개발 데이터 내에서만 다시 나눈다.

In [ ]:
validation_rows = []
tuning_start = time.perf_counter()
for run in runs:
    fold = make_fold(run["train"], team_candidate_drugs)
    val = run["validation"]
    if val.empty:
        raise ValueError("Every split needs nonempty Validation relationships.")
    targets = sorted(val["disease_id"].unique())
    for mode in FEATURE_MODES:
        neighbors = neighbors_for_targets(targets, mode, fold)
        for k in NEIGHBOR_K_OPTIONS:
            for power in SIM_POWER_OPTIONS:
                summary = evaluate(val, fold, method="strict", mode=mode, k=k, power=power, precomputed=neighbors)
                validation_rows.append({"seed":run["seed"], "feature_mode":mode,
                                        "neighbor_k":k, "sim_power":power, **summary})
validation_by_seed = pd.DataFrame(validation_rows)
validation_summary = (validation_by_seed.groupby(["feature_mode", "neighbor_k", "sim_power"], as_index=False)
                      .agg(n_splits=("seed", "nunique"),
                           mean_ndcg10=("NDCG@10", "mean"), sd_ndcg10=("NDCG@10", "std"),
                           mean_recall10=("Recall@10", "mean"), sd_recall10=("Recall@10", "std")))
validation_summary = validation_summary.sort_values(
    ["mean_ndcg10", "mean_recall10", "sim_power", "neighbor_k", "feature_mode"],
    ascending=[False,False,True,True,True],kind="stable").reset_index(drop=True)

winner = validation_summary.iloc[0]
BEST_MODE = str(winner["feature_mode"])
BEST_K = int(winner["neighbor_k"])
BEST_POWER = int(winner["sim_power"])
print("Selected from Validation only:", BEST_MODE, "k=", BEST_K, "sim_power=", BEST_POWER)
print("Total Validation runs:", len(validation_by_seed), "| Elapsed seconds:", round(time.perf_counter()-tuning_start,1))
display(validation_summary)

plt.figure(figsize=(8,4))
for mode in FEATURE_MODES:
    for p in SIM_POWER_OPTIONS:
        df_line = validation_summary[(validation_summary["feature_mode"]==mode) & (validation_summary["sim_power"]==p)].sort_values("neighbor_k")
        plt.errorbar(df_line["neighbor_k"], df_line["mean_ndcg10"], yerr=df_line["sd_ndcg10"], marker="o", capsize=3, label=f"{mode}, power={p}")
plt.xlabel("Number of neighboring diseases (k)")
plt.ylabel("Mean Validation NDCG@10 ± SD (5 splits)")
plt.title("Strict CBF: hyperparameter stability")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate Strict CBF, subset, fallback, and baseline separately
**preliminary:** 각 시드별 Train+Validation을 합쳐 각기 다른 Test에 대해 한 번씩 평가하고 **5회 평균·표준편차**를 제시한다.

**team_final:** 팀 공통 Train+Validation 전체로 재구성 후 **팀 공통 Test 1회**만 평가한다.

**주 결과는 Strict CBF (all test diseases)**. `Strict CBF (gene-profile subset)`은 유전자 정보가 있는 질병에 한정한 **보조 분석**으로 다른 팀의 전체 평가 점수와 직접 비교하지 않는다. Fallback은 CBF가 아닌 인기순 보완이 포함된 별도 모델이다.

In [ ]:
test_rows = []
all_disease_test_rows = []
first_run_examples = None
start_eval = time.perf_counter()
scored_runs = runs if RUN_MODE=="preliminary" else runs[:1]
for run in scored_runs:
    train_full = pd.concat([run["train"], run["validation"]], ignore_index=True)
    fold = make_fold(train_full, team_candidate_drugs)
    truth = run["test"]
    targets = sorted(truth["disease_id"].unique())
    neighborhood = neighbors_for_targets(targets, BEST_MODE, fold)
    strict_detail = evaluate(truth, fold, method="strict", mode=BEST_MODE, k=BEST_K, power=BEST_POWER,
                             precomputed=neighborhood, return_details=True)
    subset = strict_detail.loc[strict_detail["has_gene_profile"]].copy()
    fallback_detail = evaluate(truth, fold, method="fallback", mode=BEST_MODE, k=BEST_K, power=BEST_POWER,
                               precomputed=neighborhood, return_details=True)
    popularity_detail = evaluate(truth, fold, method="popularity", return_details=True)
    variants = [("Strict CBF - all diseases",strict_detail),
                ("Strict CBF - gene subset",subset),
                ("CBF + popularity fallback",fallback_detail),
                ("Popularity baseline",popularity_detail)]
    for method, detail in variants:
        test_rows.append({"seed":run["seed"], "method":method, **summarize_metrics(detail)})
    strict_detail.insert(0,"seed",run["seed"])
    all_disease_test_rows.append(strict_detail)
    if first_run_examples is None:
        gene_targets = [d for d in targets if feature_available[disease_to_row[d]]]
        example_disease = "DOID_10113" if "DOID_10113" in gene_targets else (gene_targets[0] if gene_targets else targets[0])
        top_neighbors = neighborhood.get(example_disease, [])
        example_rank = rank_drugs(example_disease, top_neighbors, fold, BEST_K, BEST_POWER, fallback=False)
        hidden = lookup_by_disease(truth).get(example_disease,set())
        first_run_examples = (run["seed"], example_disease, top_neighbors, example_rank, hidden)

by_seed = pd.DataFrame(test_rows)
test_summary = (by_seed.groupby("method",as_index=False)
                .agg(n_splits=("seed","nunique"),
                     eval_diseases_mean=("eval_diseases","mean"),
                     mean_recall10=("Recall@10","mean"), sd_recall10=("Recall@10","std"),
                     mean_ndcg10=("NDCG@10","mean"), sd_ndcg10=("NDCG@10","std"),
                     gene_coverage=("gene_profile_fraction","mean"),
                     recommendation_coverage=("recommendation_fraction","mean"),
                     mean_fallback_slots=("mean_fallback_slots","mean")))
method_order = ["Strict CBF - all diseases", "Strict CBF - gene subset",
                "CBF + popularity fallback", "Popularity baseline"]
test_summary["order"] = test_summary["method"].map({m:i for i,m in enumerate(method_order)})
test_summary = test_summary.sort_values("order").drop(columns="order").reset_index(drop=True)
print("Evaluation elapsed seconds:",round(time.perf_counter()-start_eval,1))
print("Protocol:", "5 separate preliminary test splits" if RUN_MODE=="preliminary" else "1 fixed team test split")
display(test_summary)

for seed, frame in by_seed.groupby("seed"):
    c = frame.loc[frame["method"]=="Strict CBF - all diseases"].iloc[0]
    print(f"Seed {seed}: Strict Recall@10={c['Recall@10']:.4f}, NDCG@10={c['NDCG@10']:.4f}, gene coverage={c['gene_profile_fraction']:.1%}")

## 8. Inspect a sample recommendation and known limitations
아래의 `hidden_answer`는 **추천 결과를 모두 만든 뒤 사후 평가를 위해 표시**한다. 모델 입력으로는 사용하지 않는다.

- 유전자 정보가 없는 질병에서는 Strict CBF가 추천할 수 없는 것이 정상이다.
- Train에 한 번도 등장하지 않은 약물은 Strict CBF에서 양의 점수를 얻기 어렵다.
- 생물학적 유전자 유사성은 약물 효과·임상 안전성의 증거가 아니다.

In [ ]:
seed, example_disease, example_neighbors, example_rank, hidden = first_run_examples
name_disease = name_maps["disease"].get(example_disease,example_disease)
neighbor_table = pd.DataFrame([{"disease_id":d,"name":name_maps["disease"].get(d,d),"cosine_similarity":round(sim,6)} for d,sim in example_neighbors[:10]])
example_table = pd.DataFrame([{"rank":idx+1,"disease_id":example_disease,"drug_id":drug,
                              "drug_name":name_maps["drug"].get(drug,drug),
                              "cbf_score":round(score,6),"score_source":method,"hidden_answer":(drug in hidden)}
                             for idx,(drug,score,method) in enumerate(example_rank[:20])])
print("Example seed:",seed,"| Disease:",name_disease, "(",example_disease,")")
print("Known hidden test drugs:",len(hidden),"| Strict CBF returned:",len(example_rank))
print("Closest diseases:")
display(neighbor_table)
print("Top 20 Strict CBF drugs:")
display(example_table)

first_train_full = pd.concat([runs[0]["train"],runs[0]["validation"]],ignore_index=True)
seen_drugs = set(first_train_full["drug_id"])
first_unseen_links = int((~runs[0]["test"]["drug_id"].isin(seen_drugs)).sum())
print("First test links to drugs unseen in the training pool:", first_unseen_links, "/",len(runs[0]["test"]))
print("Do not interpret drug rankings as clinical treatment recommendations.")

## 9. Optional secondary approved-indication evaluation (team_final only)
기존 팀 계획의 **보조 평가**를 위한 선택 기능이다. 최종 모드에서 `RUN_SECONDARY=True`일 때만 실행하며, 검증된 `team_approved_indications.csv`가 필요하다.

허가 수준은 약물 자체의 승인 여부나 Phase 숫자만으로 임의 추정하지 않는다. **주 평가와 다른 정답 집합**임을 구별해 보고한다.

In [ ]:
secondary_summary = None
if RUN_SECONDARY:
    approved = normalize_pairs(pd.read_csv(DATA_DIR / "team_approved_indications.csv"))
    if not pair_set(approved).issubset(pair_set(positive_pairs)):
        raise ValueError("Approved indication pairs must be a subset of eligible indications.")
    approved_set = pair_set(approved)
    official_test = runs[0]["test"]
    secondary_truth = official_test.loc[[tuple(p) in approved_set for p in official_test.itertuples(index=False,name=None)]]
    if secondary_truth.empty:
        raise ValueError("No approved indication pairs in the official Test split.")
    dev_full = pd.concat([runs[0]["train"],runs[0]["validation"]],ignore_index=True)
    fold = make_fold(dev_full, team_candidate_drugs)
    targets = sorted(secondary_truth["disease_id"].unique())
    neighborhood = neighbors_for_targets(targets, BEST_MODE, fold)
    strict_secondary = evaluate(secondary_truth, fold, method="strict", mode=BEST_MODE, k=BEST_K,
                                power=BEST_POWER, precomputed=neighborhood)
    base_secondary = evaluate(secondary_truth, fold, method="popularity")
    secondary_summary = pd.DataFrame([{"method":"Strict CBF (approved only)",**strict_secondary},
                                      {"method":"Popularity (approved only)",**base_secondary}])
    print("Approved Test links:",len(secondary_truth),"| Diseases:",len(targets))
    display(secondary_summary[["method","eval_diseases","Recall@10","NDCG@10"]])
else:
    print("Secondary evaluation not requested; no approved-indication status inferred.")

## 10. Save reproducible summaries (no full raw dataset export)
실험 설정, 선택된 파라미터, 5개 분할별 Validation 결과, 초기 Test 결과와 추천 예시를 로컬에 저장한다.

- `preliminary` 결과는 **최종 팀 비교 성능이 아님**.
- `team_final` 결과는 공통 정답/후보 목록/분할 파일의 **정확한 일치**를 전제로 한다.
- 저장한 관계 기반 분할이나 전체 원본 CSV는 라이선스 확인 전 GitHub에 올리지 않는다. 팀에 공유할 때는 노트북·집계 지표를 우선 사용한다.

In [ ]:
validation_by_seed.to_csv(OUTPUT_DIR/"cbf_validation_by_seed.csv",index=False)
validation_summary.to_csv(OUTPUT_DIR/"cbf_validation_summary.csv",index=False)
by_seed.to_csv(OUTPUT_DIR/"cbf_test_by_seed.csv",index=False)
test_summary.to_csv(OUTPUT_DIR/"cbf_test_summary.csv",index=False)
example_table.to_csv(OUTPUT_DIR/"cbf_example_recommendations.csv",index=False)
if secondary_summary is not None:
    secondary_summary.to_csv(OUTPUT_DIR/"cbf_approved_secondary.csv",index=False)

meta = {"run_mode":RUN_MODE,"label_status":label_status,"catalog_policy":catalog_status,
        "seed_values":SEEDS,"feature_modes":FEATURE_MODES,"k_options":NEIGHBOR_K_OPTIONS,
        "sim_power_options":SIM_POWER_OPTIONS,"selection_metric":SELECTION_METRIC,
        "selected_feature_mode":BEST_MODE,"selected_k":BEST_K,"selected_sim_power":BEST_POWER,
        "eligible_pairs":len(positive_pairs),"original_pairs":len(original_pairs),
        "test_protocol":"five_distinct_tests" if RUN_MODE=="preliminary" else "single_fixed_team_test",
        "primary_method":"Strict CBF (all diseases)","secondary_approved_enabled":RUN_SECONDARY,
        "note":"Not a clinical efficacy assessment."}
(OUTPUT_DIR/"cbf_run_metadata.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
print("Output files:")
for file in sorted(OUTPUT_DIR.glob("cbf_*")):
    print("-",file.name)
print("Selected setting:",BEST_MODE,"k=",BEST_K,"power=",BEST_POWER)
print("Primary method: Strict CBF - all diseases")
print("Experiment status:","PRELIMINARY - NOT FINAL" if RUN_MODE=="preliminary" else "TEAM FINAL - check cross-model evaluation agreement")

## 11. Interpretation and checklist for team sharing

**결과를 보고할 때** 아래 항목을 분리해 적는다.

- **대표 결과:** `Strict CBF - all diseases`의 Recall@10 / NDCG@10 (예비 모드에서는 5회 평균 ± 표준편차).
- **유전자 정보의 한계:** `Strict CBF - gene subset`은 평가 대상이 다르므로 PPR/Louvain 전체 질병 성능과 직접 비교하지 않는다.
- **Popularity fallback:** 빈 순위를 인기도로 채운 혼합 추천. Strict CBF와 합쳐 한 모델 성능처럼 보고하지 않는다.
- **파라미터 근거:** `raw`/`idf`, k, 유사도 지수 p를 *5개 Validation*에서 비교한 평균 및 표준편차.
- **데이터 누수 방지:** 모델이 참조하는 적응증은 Train뿐이고, Test는 점수 계산·모델 선택에 사용하지 않는다. 예비 후보 약물은 각 분할의 Train 기준이고, 최종 후보 약물은 사전 확정한 공통 파일로 고정한다.
- **팀 공통 조건:** 최종 정제 57,835쌍, 공통 분할, 공통 후보 약물과 평가 질병, Recall@K/NDCG@K 정의, Popularity fallback 적용 여부.
- **추가 한계:** 질병 유전자 정보 결측, Train에 없는 약물, 기록된 적응증 자체의 임상·출처 불확실성, 유전자 연관 정보의 간접적 라벨 누수 가능성, 수업에서 배운 전형적 CBF와의 구조적 차이.

**주의:** 예비 분할 5회는 서로 겹칠 수 있는 다른 Train/Test 분할 5개이지 독립적인 임상 검증이 아니다. 최종 모드의 5회는 고정된 Test 외부의 *내부 Validation 5회*이며, Test를 다섯 번 평가한 것이 아니다.

**출처:** OptimusKG V3.2, Harvard Dataverse DOI 10.7910/DVN/IYNGEV; SCE331 DM_09-Recom1, Content-Based Recommendation (Feature Profiles / Similarity).